In [55]:
# load libaries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as sm

The tool outputs are created by the script "merge_tool_outputs.ipynb".\
The tools have max capacity and therefore the data set with antibodies had to be split up for predictions to be performed. To reduce risk of intefering with the merging of the files, to make each script shorter and easier to understand and to be able to remove the originals files this computation of scores part and the merge tool outputs part have been separated. 

In [56]:
# Load merged tool outputs
netMHC1_EL_pep9 = pd.read_csv("merged_tool_outputs/only_useful_scores/netMHC1_EL_pep9_only_useful_scores.csv")
netMHC1_EL_pep14 = pd.read_csv("merged_tool_outputs/only_useful_scores/netMHC1_EL_pep14_only_useful_scores.csv")
netMHC_II_EL_pep15 = pd.read_csv("merged_tool_outputs/only_useful_scores/netMHC_II_EL_pep15_only_useful_scores.csv")
netMHC_II_EL_pep12 = pd.read_csv("merged_tool_outputs/only_useful_scores/netMHC_II_EL_pep12_only_useful_scores.csv")
waltz = pd.read_csv("merged_tool_outputs/only_useful_scores/waltz_merged_tool_outputs.csv")
biophi_relaxed = pd.read_csv("merged_tool_outputs/only_useful_scores/biophi_relaxed_only_useful_scores.csv")
biophi_strict = pd.read_csv("merged_tool_outputs/only_useful_scores/biophi_strict_only_useful_scores.csv")


# Compute scores
netMHCpan tools give one score for each peptide - HLA allele combination
Therefore, for each antibody a immunogenicity score will be calculated. The definition of what score is considerd immunogenetic differs for the different tools \
for each netMHC score of interest: precentile score, immunogenicity socre and pre processing score

In [75]:
# netMHC1_EL_pep9 percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep9_percentile = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep9_percentile')
    )

# netMHC1_EL_pep9 Immunogenicity score 

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
netMHC1_pep9_immunogenicity_score = (
    netMHC1_EL_pep9.assign(immunogenic=netMHC1_EL_pep9['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep9_immunogenicity_score')
    )

# netMHC1_EL_pep9 Preprocessing score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMCH1_pep9_preProcess = netMHC1_EL_pep9.groupby('antibody')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'netMHC1_pep9_preProcess'})

# netMHC1_EL_pep14

# Percentile score

# Immunogenentic is defened as scored <= 1%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 1. 
netMHC1_pep14_percentile = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['netmhcpan_el percentile'] <= 1) # flags rows where percentile is below 1
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep14_percentile')
    )

# Immunogenicity score

# Immunogenentic is defened as scored larger than 0
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a immunogenicity score above 0. 
netMHC1_pep14_immunogenicity_score = (
    netMHC1_EL_pep14.assign(immunogenic=netMHC1_EL_pep14['immunogenicity score'] > 0) # flags rows where percentile is above 0
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC1_pep14_immunogenicity_score')
    )

# Pre-proocessing score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 

netMCH1_pep14_preProcess = netMHC1_EL_pep14.groupby('antibody')['processing total score'].mean().reset_index().rename(columns={'processing total score': 'netMHC1_pep14_preProcess'})


# netMHC_II_EL_pep12

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep12_percentile = (
    netMHC_II_EL_pep12.assign(immunogenic=netMHC_II_EL_pep12['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep12_percentile')
    )

# Immunogenicity score

# remove the rows with the immunogenicity score of '-' before calculating the mean
netMHC_II_EL_pep12 = netMHC_II_EL_pep12[netMHC_II_EL_pep12['immunogenicity score'] != '-']
# make the column with the immunogenicity score into a numeric column
netMHC_II_EL_pep12['immunogenicity score'] = pd.to_numeric(netMHC_II_EL_pep12['immunogenicity score'])

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody.
netMHC_II_pep12_immunogenicity_score = netMHC_II_EL_pep12.groupby('antibody')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'netMHC_II_pep12_immunogenicity_score'})


# Does not have pre processing score. Due to tool not being able to predict pre processing on peptides shorter than 13 amino acids


# netMHC_II_EL_pep15

# Percentile score

# Immunogenentic is defened as scored <= 10%
# Here I calculate the percantage of peptide-HLA allele combinations (rows) that have a percentile score below 10. 
netMHC_II_pep15_percentile = (
    netMHC_II_EL_pep15.assign(immunogenic=netMHC_II_EL_pep15['netmhciipan_el percentile'] <= 10) # flags rows where percentile is below 10
          .groupby('antibody')['immunogenic'] # calculates mean of immunogenic for each antibody, gives the fraction
          .mean()
          .mul(100) # multiplies by 100 to get percentage
          .reset_index(name='netMHC_II_pep15_percentile')
    )

# Immunogenicity score

# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_immunogenicity_score = netMHC_II_EL_pep15.groupby('antibody')['immunogenicity score'].mean().reset_index().rename(columns={'immunogenicity score':'netMHC_II_pep15_immunogenicity_score'})

# Pre-proocessing score
# MHC class 2 has 2 preprocessing scores of interest: mhcii-np cleavage probability score and mhcii-np cleavage probability percentile rank

# mhcii-np cleavage probability

# remove the rows with the cleavage probability score of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability score'] != '-']
# make the column with the cleavage probability score into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability score'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability score'])
# Compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_preProcess_cleavProb = netMHC_II_EL_pep15.groupby('antibody')['mhcii-np cleavage probability score'].mean().reset_index().rename(columns={'mhcii-np cleavage probability score': 'netMHC_II_pep15_preProcess_cleavProb'})

# mhcii-np cleavage probability percentile rank

# remove the rows with the cleavage probability percentile rank of '-' before calculating the mean
netMHC_II_EL_pep15 = netMHC_II_EL_pep15[netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] != '-']
# make the column with the cleavage probability percentile rank into a numeric column
netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'] = pd.to_numeric(netMHC_II_EL_pep15['mhcii-np cleavage probability percentile rank'])
# compute score
# Immunogenentic is not defined
# Here I simply calculate the mean score for each antibody. 
netMHC_II_pep15_preProcess_cleavProbPercentile = netMHC_II_EL_pep15.groupby('antibody')['mhcii-np cleavage probability percentile rank'].mean().reset_index().rename(columns={'mhcii-np cleavage probability percentile rank': 'netMHC_II_pep15_preProcess_cleavProbPercentile'})


# waltz

# rename the column '...' to nr_aggs
waltz = waltz.rename(columns={'...': 'nr_aggs'})

# compute the total number of amio acids that are considerd immunogenetic
def sum_ranges(s):
    if pd.isna(s):
        return 0
    total = 0
    for part in s.split(';'):
        start, end = part.strip().split('-')
        total += int(end) - int(start) + 1 # beacuse the values are inclusive
    return total

waltz['nr_aggs'] = waltz['waltz_score'].apply(sum_ranges)

# biophi 
# only rename the antibody column so it matches the rest
biophi_relaxed = biophi_relaxed.rename(columns={'Antibody': 'antibody'})
biophi_strict = biophi_strict.rename(columns={'Antibody': 'antibody'})

# Load ADA scores + add one previously predicted antibody


In [80]:

ADA = pd.read_csv("ADA_Clinical_Ab_2021_biophiDataset.csv")
# extract only the ADA (here called Immunogenicity) and Antibody (name) columns
ADA = ADA[["Antibody", "Immunogenicity"]].copy()
# make all antibody names lowercase
ADA["Antibody"] = ADA["Antibody"].str.lower()
# rename antibody and immunogenicity column
ADA = ADA.rename(columns={'Antibody': 'antibody', 'Immunogenicity': 'ADA_percentage'})


# load another dataset with antibodies, that have been predicited previously
previous_37AB = pd.read_csv("../Antibodies/ADA_combined_features.csv")
# make all antibody names lowercase
previous_37AB["antibody"] = previous_37AB["antibody"].str.lower()
# check which antibodies in previous_37AB does not exist in ADA
antibodies_not_in_ADA = set(previous_37AB['antibody']) - set(ADA['antibody'])
print("Antibodies in previous_37AB that are not in ADA:", antibodies_not_in_ADA)    
# Will add this antibody in the merging step


Antibodies in previous_37AB that are not in ADA: {'visilizumab'}


# Merge scores from all predictors into one df

In [81]:
netMHC_II_pep15_preProcess_cleavProb

,antibody,netMHC_II_pep15_preProcess_cleavProb
0,3f8,0.114987
1,abagovomab,0.120447
2,abciximab,0.119223
3,abrilumab,0.146311
4,actoxumab,0.095804
...,...,...
212,veltuzumab,0.096205
213,xentuzumab,0.128650
214,zalutumumab,0.117738
215,zolbetuximab,0.111857


In [84]:


# create df with all the dfs that shall me merged into one
dfs_to_merge = [
    netMHC1_pep9_percentile,
    netMHC1_pep9_immunogenicity_score,
    netMCH1_pep9_preProcess,
    netMHC1_pep14_percentile,
    netMHC1_pep14_immunogenicity_score,
    netMCH1_pep14_preProcess,
    netMHC_II_pep12_percentile,
    netMHC_II_pep12_immunogenicity_score,
    netMHC_II_pep15_percentile,
    netMHC_II_pep15_immunogenicity_score,
    netMHC_II_pep15_preProcess_cleavProb,
    netMHC_II_pep15_preProcess_cleavProbPercentile,
]

# Rename the sequence name column to antibody to make the merging easier. 
dfs_to_merge = [df.rename(columns={'sequence name': 'antibody'}) for df in dfs_to_merge]

# Create the df with all scores, start with the ADA scores
all_predictors_andADA = ADA

# Merge all dfs on antibody name
for df in dfs_to_merge:
    all_predictors_andADA = all_predictors_andADA.merge(df, on='antibody', how='left')

# remove all whitespaces in antobdy column
all_predictors_andADA['antibody'] = all_predictors_andADA['antibody'].str.strip()

# Then merge the three last dfs
all_predictors_andADA = all_predictors_andADA.merge(waltz[['antibody','nr_aggs']], on='antibody', how='left').rename(columns={'nr_aggs': 'waltz_nr_aggs'})
all_predictors_andADA = all_predictors_andADA.merge(biophi_relaxed, on='antibody', how='left').rename(columns={'OASis Identity': 'biophi_KabKabRelaxed_score'})
all_predictors_andADA = all_predictors_andADA.merge(biophi_strict, on='antibody', how='left').rename(columns={'OASis Identity': 'biophi_KabKabStrict_score'})

# Add the row visilizumab from the previous_37AB df to the all_predictors_andADA, they should have the same columns

# add the row with the antibody visilizumab from the previous_37AB df to the all_predictors_andADA df, 
# but only if it does not already exist in all_predictors_andADA

visilizumab_row = previous_37AB[previous_37AB['antibody'] == 'visilizumab']
all_predictors_andADA = pd.concat([all_predictors_andADA, visilizumab_row], ignore_index=True)


all_predictors_andADA

,antibody,ADA_percentage,netMHC1_pep9_percentile,netMHC1_pep9_immunogenicity_score,netMHC1_pep9_preProcess,netMHC1_pep14_percentile,netMHC1_pep14_immunogenicity_score,netMHC1_pep14_preProcess,netMHC_II_pep12_percentile,netMHC_II_pep12_immunogenicity_score,netMHC_II_pep15_percentile,netMHC_II_pep15_immunogenicity_score,netMHC_II_pep15_preProcess_cleavProb,netMHC_II_pep15_preProcess_cleavProbPercentile,waltz_nr_aggs,biophi_KabKabRelaxed_score,biophi_KabKabStrict_score
0,3f8,100.0,3.445306,37.674419,-3.373804,0.229277,38.095238,-3.531327,8.010336,99.118753,10.298103,94.751924,0.114987,29.354146,51,0.497585,0.188406
1,abagovomab,68.1,2.837241,44.495413,-3.417786,0.243436,41.314554,-3.561854,7.838071,99.484626,8.465608,95.632836,0.120447,24.888571,53,0.433333,0.180952
2,abciximab,35.5,3.227999,39.908257,-3.443564,0.208659,37.558685,-3.587375,8.354866,99.494547,8.024691,96.265295,0.119223,26.229048,23,0.428571,0.147619
3,abrilumab,0.4,2.508961,39.170507,-3.543928,0.192173,35.377358,-3.673514,7.760141,99.620457,8.581752,96.304780,0.146311,29.753171,30,0.985646,0.837321
4,actoxumab,0.0,2.966315,44.343891,-3.409749,0.240055,41.203704,-3.548555,7.996633,99.447036,7.583774,96.624264,0.095804,31.544048,39,0.924883,0.835681
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,zalutumumab,0.0,3.191138,42.410714,-3.394748,0.219855,38.356164,-3.521994,6.584362,99.248820,7.665805,95.634202,0.117738,25.100465,42,0.930556,0.805556
214,zolbetuximab,0.0,3.039362,39.461883,-3.458596,0.220863,36.238532,-3.593007,6.804479,99.524084,9.302326,96.309374,0.111857,27.603953,44,0.562791,0.334884
215,zolimomab,85.7,3.618365,33.179724,-3.398370,0.296995,31.603774,-3.558859,6.287683,99.571700,7.859079,95.456749,0.157254,25.740732,45,0.373206,0.095694
216,moab_81c6,100.0,3.686945,42.081448,-3.399031,0.257202,39.814815,-3.552895,8.585859,99.434207,9.171076,95.339264,0.152478,26.621190,49,0.516432,0.352113


In [85]:
# Make all white space and '-' in the column names to '_'
all_predictors_andADA.columns = (
    all_predictors_andADA.columns
    .str.replace(r'[\s\-]+', '_', regex=True)
)

In [86]:
# export score table
all_predictors_andADA.to_csv("all_predictors_217AB(biophidata).csv")